In [ ]:
import pandas as pd
import matplotlib
from pathlib import Path
from IPython.display import Image, display
from risk_validation.core.metrics.impl import pd as pd_metrics  # noqa: F401 — registers PD metrics
from risk_validation.core.services.pd.metrics_service import PDMetricsService
from risk_validation.core.utils.plots import plot_roc, plot_gini, plot_ks_cdf_with_maxgap

# Set data directory
DATA_DIR = Path('..') / 'data'
csv_path = DATA_DIR / 'sample.csv'

# Load data
try:
    df = pd.read_csv(csv_path)
except FileNotFoundError:
    raise FileNotFoundError(f"CSV file not found at {csv_path}")

# Define column mappings from authoritative context
params = {
    'y_col': 'default_flag',
    'p_col': 'pred_br',
    'score_col': 'score_pd',
    'period_col': 'QTR',
    'band_col': 'BAND'
}

# Extract dev and val data
available_years = df['score_year'].unique()
dev_year, val_year = 2019, 2020

# Filter dataframes for development and validation periods
assert dev_year in available_years and val_year in available_years, "Specified years are not in the dataset."
df_dev = df[df['score_year'] == dev_year]
df_val = df[df['score_year'] == val_year]

# Initialize metrics service
service = PDMetricsService(['gini', 'auc_roc', 'ks'])

# Compute metrics without period_col as we filter by year
result_dev = service.compute(df_dev, params)
result_val = service.compute(df_val, params)

# Ensure computations returned 'value'
if not ('value' in result_dev.columns and 'value' in result_val.columns):
    raise ValueError("Metrics computation failed, 'value' column is missing.")

# Print results
print("Development Metrics:", result_dev[['metric_id', 'value']])
print("Validation Metrics:", result_val[['metric_id', 'value']])

# Plotting
matplotlib.use("Agg")  # Use non-GUI backend for safe plot generation
plot_dir = Path('plots')
plot_dir.mkdir(exist_ok=True)

# ROC Plot
roc_path = plot_roc(
    y_true=df_val['default_flag'],
    y_score=df_val['pred_br'],
    title='ROC Curve - KS Comparison',
    out_path=plot_dir / 'roc.png'
)
# Display ROC plot
display(Image(filename=str(roc_path)))

# Gini/Gains Plot
gini_path = plot_gini(
    y_true=df_val['default_flag'],
    y_score=df_val['pred_br'],
    title='Gini / Gains Curve Comparison',
    out_path=plot_dir / 'gini.png'
)
# Display Gini Plot
display(Image(filename=str(gini_path)))

# KS CDF Plot
ks_path, ks_table = plot_ks_cdf_with_maxgap(
    y_true=df_val['default_flag'],
    y_score=df_val['pred_br'],
    title='KS Statistic CDF Plot',
    out_path=plot_dir / 'ks_cdf.png'
)
# Display KS CDF Plot
display(Image(filename=str(ks_path)))